In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow 버전:', tf.__version__)
print('NumPy 버전:', np.__version__)
print('Pandas 버전:', pd.__version__)

TensorFlow 버전: 2.10.0
NumPy 버전: 1.23.5
Pandas 버전: 2.3.3


In [3]:
# sigmoid이전의 선형 출력값 예시
H_example = np.array([[-3.0], [-1.0], [0.0], [1.0], [3.0]])
H_example

array([[-3.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 3.]])

In [4]:
z_example = tf.sigmoid(H_example)
z_example

<tf.Tensor: shape=(5, 1), dtype=float64, numpy=
array([[0.04742587],
       [0.26894142],
       [0.5       ],
       [0.73105858],
       [0.95257413]])>

In [5]:
# H, z비교
sigmoid_result_df = pd.DataFrame({
    # (5,1)모양을 1차원으로 피기
    'H': H_example.reshape(-1),
    # z_example은 tensor이므로 numpy배열로 만들기
    'z = sigmoid(H)': z_example.numpy().reshape(-1)
})
sigmoid_result_df

,H,z = sigmoid(H)
0,-3.0,0.047426
1,-1.0,0.268941
2,0.0,0.500000
3,1.0,0.731059
4,3.0,0.952574


In [6]:
# 확률이 0.5넘는지 true / false 를 1 / 0으로 cast
prediction_example = tf.cast(z_example >= 0.5, tf.int32)
sigmoid_result_df['prediction'] = prediction_example.numpy().reshape(-1)
sigmoid_result_df

,H,z = sigmoid(H),prediction
0,-3.0,0.047426,0
1,-1.0,0.268941,0
2,0.0,0.500000,1
3,1.0,0.731059,1
4,3.0,0.952574,1


In [7]:
# sequential 모델 생성 Dense(1) = Linear(1,1)
# 랜덤 초기값 고정
tf.random.set_seed(42)
model = tf.keras.Sequential([
    # 입력이 데이터 한개당 특성(feature) 1개
    # (2,) 하면 ax + bx1 + c 됨
    # (1,1) 하면 입력데이터 차원 늘어나서 [[1], [2]] 이게 [[[1]], [[2]]]이런식으로 됨
    tf.keras.Input(shape=(1,)),

    # fully connected layer. 출력(뉴런) 1개
    # activation='sigmoid' 안넣음. binarycrossentropy(from_logits=True)에서 H를 요구하기 때문
    tf.keras.layers.Dense(1)
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________


In [8]:
X = np.array([160, 170, 180, 190], dtype=np.float32).reshape(-1,1)
y = np.array([0, 0, 1, 1], dtype=np.float32).reshape(-1,1)

X_mean = X.mean()
X_std = X.std()
X_norm = (X - X_mean) / X_std

X_mean, X_std, X_norm, X_norm.shape

(175.0,
 11.18034,
 array([[-1.3416408],
        [-0.4472136],
        [ 0.4472136],
        [ 1.3416408]], dtype=float32),
 (4, 1))

In [9]:
# 학습 전 model 출력 확인
# z가 아니라 H값 필요
H_before = model(X_norm)
# 확률 보려면 직접 지정해야함
z_before = tf.sigmoid(H_before)
print('학습 전 H 예시:')
print(H_before.numpy()[:5])

print('\n학습 전 sigmoid(H), 즉 z 예시:')
print(z_before.numpy()[:5])

학습 전 H 예시:
[[-0.3597958 ]
 [-0.11993194]
 [ 0.11993194]
 [ 0.3597958 ]]

학습 전 sigmoid(H), 즉 z 예시:
[[0.41100898]
 [0.47005293]
 [0.5299471 ]
 [0.58899105]]


In [10]:
# 새시작. sequential 모델 새로 생성
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(1)
])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________


In [11]:
# 데이터도 새로 준비
X = np.array([160, 170, 180, 190], dtype=np.float32).reshape(-1, 1)
y = np.array([0, 0, 1, 1], dtype=np.float32).reshape(-1, 1)
X, y, X.shape, y.shape

(array([[160.],
        [170.],
        [180.],
        [190.]], dtype=float32),
 array([[0.],
        [0.],
        [1.],
        [1.]], dtype=float32),
 (4, 1),
 (4, 1))

In [12]:
X_mean = X.mean()
X_std = X.std()
X_norm = (X - X_mean) / X_std

X_mean, X_std, X_norm, X_norm.shape

(175.0,
 11.18034,
 array([[-1.3416408],
        [-0.4472136],
        [ 0.4472136],
        [ 1.3416408]], dtype=float32),
 (4, 1))

In [ ]:
# model에 입력값 넣으면 H(선형 계산값) 반환
H_before = model(X_norm)
z_before = tf.sigmoid(H_before)
print('학습 전 H 예시:')
print(H_before.numpy()[:5])

print('\n학습 전 sigmoid(H), 즉 z 예시:')
print(z_before.numpy()[:5])

학습 전 H 예시:
[[ 1.8400794]
 [ 0.6133598]
 [-0.6133598]
 [-1.8400794]]

학습 전 sigmoid(H), 즉 z 예시:
[[0.86295813]
 [0.64870685]
 [0.35129315]
 [0.1370419 ]]


In [ ]:
learning_rate = 0.1
epochs = 1000
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
    # H를 입력으로 받는 binary cross entropy
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True)
)
learning_rate, epochs

(0.1, 1000)

In [15]:
# model.fit 학습
# 정규화된 값 사용
# 자동으로 H계산 -> Cost계산-> gradient 계산 -> weight/bias 업데이트
# verbose=0 진행막대 숨기기
# 기본 batch_size는 32. 예제가 4개뿐이므로 생략
history = model.fit(
    X_norm,
    y,
    epochs=epochs,
    verbose=0
)
print('학습 완료!')
list(history.history.keys())

학습 완료!


['loss']

In [ ]:
# history['loss']에서 cost 계산과정 확인
for epoch_index, mean_cost in enumerate(history.history['loss']):
    if epoch_index % 100 == 0 or epoch_index == epochs - 1:
        print(f'epoch={epoch_index}, mean_cost={mean_cost:.6f}')

final_mean_cost = history.history['loss'][-1]
print(f'\n최종 mean_cost(평균 Cost): {final_mean_cost:.6f}')

epoch=0, mean_cost=1.516801
epoch=100, mean_cost=0.231581
epoch=200, mean_cost=0.144888
epoch=300, mean_cost=0.110192
epoch=400, mean_cost=0.090081
epoch=500, mean_cost=0.076558
epoch=600, mean_cost=0.066705
epoch=700, mean_cost=0.059155
epoch=800, mean_cost=0.053162
epoch=900, mean_cost=0.048278
epoch=999, mean_cost=0.044254

최종 mean_cost(평균 Cost): 0.044254


In [17]:
# weight, bias 확인
# 모델 마지막층(Dense(1))에 학습된 weight, bias가 들어 있음
weight, bias = model.layers[-1].get_weights()

# weight는 (입력특성수, 출력수) - (1, 1) 형태의 2차원 배열이라 두번 인덱싱해야됨
# bias는 (출력수, ) - (1, )형태의 1차원 배열이라 한번만 인덱싱 해도 됨

a = weight[0][0]
b = bias[0]

a, b, weight, bias

(5.3443875,
 -1.3038517e-08,
 array([[5.3443875]], dtype=float32),
 array([-1.3038517e-08], dtype=float32))

In [18]:
# 학습 후 H와 probability확인
H = model(X_norm)

probability = tf.sigmoid(H)
H, probability

(<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
 array([[-7.1702485],
        [-2.3900828],
        [ 2.3900828],
        [ 7.1702485]], dtype=float32)>,
 <tf.Tensor: shape=(4, 1), dtype=float32, numpy=
 array([[7.6854049e-04],
        [8.3932064e-02],
        [9.1606790e-01],
        [9.9923146e-01]], dtype=float32)>)

In [ ]:
# 새 입력값 예측
input_value = 185
input_norm = (input_value - X_mean) / X_std
# Dense(1)에 넣으려면 shape를 (1,1)로 맞춰야하므로
input_norm_array = np.array([[input_norm]], dtype=np.float32)
# 대용량 데이터 예측할때는 model.predict()사용하는게 맞지만 데이터양이 적다면 model() 안에 넣어서 __call__메서드 호출하는 방식이 훨씬 빠름
H_new = model(input_norm_array)
# 확률 확인
# 예측시에는 sigmoid를 직접 적용해서 z를 구함
probability = tf.sigmoid(H_new)

prediction = 1 if probability.numpy().item() >= 0.5 else 0
input_value, input_norm, H_new.numpy().item(), probability.numpy().item(), prediction

(185, 0.8944271969412381, 4.780165672302246, 0.9916752576828003, 1)

In [20]:
print(f'키가 {input_value}cm인 사람의 농구선수 확률(probability): {probability.numpy().item():.4f}')
if prediction == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

키가 185cm인 사람의 농구선수 확률(probability): 0.9917
판별 결과: 농구선수입니다.
